# Give field teams a list of points

> Export a plan as a CSV file with latitude and longitude

::: {.callout-note appearance="simple" icon=false}
**How-to guide: get one job done.** It assumes that you know the basics. New to trufl? Take the [tutorial](../tutorials/01_getting_started.ipynb) first. [About the four kinds of page](../explanation/06_documentation.ipynb).
:::

**You need:** `areas`, and the number of samples in each, `n`, in the same order. If you already have a `plan`, as after [closing a zone](05_closed_zones.ipynb), start at step 2.

In [ ]:
#| hide
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
from fastcore.test import test_eq

from trufl.utils import gridder
from trufl.callbacks import State, MaxCB, MinCB, StdCB, CountCB, PriorCB
from trufl.optimizer import Optimizer
from trufl.sampler import Sampler, rank_to_sample
from trufl.collector import DataCollector

fname_raster = '../files/ground-truth-01-4326-simulated.tif'
areas = gridder(fname_raster)
no_samples = gpd.GeoDataFrame(index=pd.Index([], name='loc_id'), geometry=None, data={'value': None})
prior_state = State(no_samples, areas, cbs=[PriorCB(fname_raster)])()
first_ranks = Optimizer(prior_state).get_rank(is_benefit_x=[True], w_vector=[1], s_method='CP')
first_n = rank_to_sample(first_ranks['rank'].reindex(areas.index).to_numpy(), budget=600, min=1, policy='quantiles')
measurements = DataCollector(fname_raster).collect(
    Sampler(areas).sample(first_n, method='uniform', rng=np.random.default_rng(0)))
state = State(measurements, areas, cbs=[MaxCB(), MinCB(), StdCB(), CountCB(), PriorCB(fname_raster)])()
ranks = Optimizer(state).get_rank(is_benefit_x=[True, True, True, False, True],
                                  w_vector=[0.25, 0.125, 0.125, 0.25, 0.25], n_method='LINEAR1', s_method='CP')

import tempfile
from pathlib import Path
n = rank_to_sample(ranks['rank'].reindex(areas.index).to_numpy(), budget=400, min=1, policy='quantiles')
csv_file = Path(tempfile.mkdtemp())/'plan.csv'

## 1. Place the points

Note the seed with the plan, so that you can make it again.

In [ ]:
import numpy as np
from trufl.sampler import Sampler

plan = Sampler(areas).sample(n, method='uniform', rng=np.random.default_rng(0))

## 2. Make one row per point

In [ ]:
points = (plan.explode(index_parts=False).to_crs(4326).get_coordinates()
          .rename(columns={'x': 'lng', 'y': 'lat'})[['lat', 'lng']]
          .round(6).reset_index())
points.insert(0, 'point', range(1, len(points) + 1))
points.head()

,point,loc_id,lat,lng
0,1,0,43.262873,-1.213818
1,2,1,43.269651,-1.222877
2,3,2,43.286865,-1.211138
3,4,3,43.294279,-1.214279
4,5,4,43.305222,-1.215237


## 3. Save it

In [ ]:
points.to_csv(csv_file, index=False)

In [ ]:
#| hide
test_eq(len(pd.read_csv(csv_file)), n.sum())

## 4. Send it with instructions

- what to do if a point cannot be reached, for example move to the nearest safe place in the same area;
- to record the position where the sample was really taken;
- the sampling depth and method.

If you import into DSS4NAFA, check the file on a few rows first. It may expect more columns, and this export has not been tested against it.

When the results come back: [add them to the campaign](07_add_results.ipynb).